# Comparison of the three approaches

1. **Conventional XGBoost** - hand-built features, Optuna-tuned (100 trials).
2. **Autoresearch, shared features** (`autoresearch/`, tag `run2`) - agent edits features and model on the same 173 processed features.
3. **Autoresearch, raw line items** (`autoresearch_raw_data/`, tag `run1`) - agent builds every feature from first-invoice line items.
4. **Autoresearch, XGBoost only** (`autoresearch_xgboost_only/`, tag `run1`) - same as 2, but the model must be XGBoost.

**CV AUC** is the mean validation AUC-ROC over four time-ordered folds (what the agents optimised). **Test AUC-ROC** is the single score on the held-out test set (the final result).

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 200)

ROOT = Path.cwd().resolve()
if ROOT.name == 'results':
    ROOT = ROOT.parent
RESULTS = ROOT / 'results'

APPROACHES = {
    'Conventional XGBoost': dict(json='conventional_xgboost.json', folder=None),
    'Autoresearch: shared features': dict(json='autoresearch_shared_features.json', folder='autoresearch'),
    'Autoresearch: raw line items': dict(json='autoresearch_raw_data.json', folder='autoresearch_raw_data'),
    'Autoresearch: XGBoost only': dict(json='autoresearch_xgboost_only.json', folder='autoresearch_xgboost_only'),
}
COLORS = {
    'Autoresearch: shared features': '#0072B2',
    'Autoresearch: raw line items': '#D55E00',
    'Autoresearch: XGBoost only': '#009E73',
}

res = {name: json.loads((RESULTS / a['json']).read_text()) for name, a in APPROACHES.items()}
logs = {}
for name, a in APPROACHES.items():
    if a['folder']:
        df = pd.read_csv(ROOT / a['folder'] / 'results.tsv', sep='\t')
        df['best_so_far'] = df['CV_AUC'].where(df['status'] == 'keep').cummax().ffill()
        logs[name] = df
{n: len(d) for n, d in logs.items()}

## 1. Summary table

In [ ]:
rows = []
for name, r in res.items():
    t = r['test_metrics']
    rows.append({
        'Approach': name,
        'Best CV AUC': r['cv']['best_mean_auc'],
        'Test AUC-ROC': t['auc_roc'],
        'Test F1': t['f1'],
        'Test precision': t['precision'],
        'Test recall': t['recall'],
        'Features': r['n_features'],
        'Experiments / trials': r.get('n_experiments', r.get('n_trials')),
    })
summary = pd.DataFrame(rows).set_index('Approach')
summary['CV - test gap'] = summary['Best CV AUC'] - summary['Test AUC-ROC']
summary.round(4)

Test AUC-ROC is the headline number. Test F1, precision and recall use a fixed 0.5 threshold, so they depend on how well each model is calibrated and are not directly comparable. The test set has 1,009 customers, so differences of a few thousandths in AUC are within noise. The `CV - test gap` column shows how much of each CV score did not carry over to unseen customers.

## 2. Best model, hyperparameters and variables per approach

### Conventional XGBoost

In [ ]:
r = res['Conventional XGBoost']
print('Tuner:', r['tuner'], '|', r['n_trials'], 'trials | seed', r['random_seed'])
print('Features:', r['n_features'], '(the shared processed table, data/processed/train.csv)')
pd.Series(r['best_params'], name='best value').to_frame()

### Autoresearch approaches

For each one: the best kept experiment (highest CV AUC among `keep` rows), with the model and the variable list as the agent logged them. The `variables` column is written by hand by the agent, so the number of names in it can differ slightly from `n_features` in the final result file, which is the authoritative count.

In [ ]:
import textwrap

best_rows = {}
for name, df in logs.items():
    kept = df[df['status'] == 'keep']
    best = kept.loc[kept['CV_AUC'].idxmax()]
    best_rows[name] = best
    names = [v.strip() for v in str(best['variables']).split(',') if v.strip()]
    print('=' * 100)
    print(name)
    print(f"Best experiment: #{best['experiment']}  CV AUC {best['CV_AUC']:.4f}  (test AUC {res[name]['test_metrics']['auc_roc']:.4f})")
    print('Model:', best['model'])
    print('Change at this experiment:', best['modification'])
    print(f"Variables logged: {len(names)}   |   n_features in result file: {res[name]['n_features']}")
    print(textwrap.fill(', '.join(names), 110))
    print()

Final `train.py` of each autoresearch approach (the code the test evaluation used):

In [ ]:
for name, a in APPROACHES.items():
    if a['folder']:
        print('#' * 100)
        print('#', name, '->', a['folder'] + '/train.py')
        print('#' * 100)
        print((ROOT / a['folder'] / 'train.py').read_text())

## 3. How the CV AUC changed over the experiments

In [ ]:
start = []
for name, df in logs.items():
    kept = df[df['status'] == 'keep']
    start.append({
        'Approach': name,
        'Experiment 1 (baseline)': df['CV_AUC'].iloc[0],
        'Best kept CV AUC': kept['CV_AUC'].max(),
        'Gain': kept['CV_AUC'].max() - df['CV_AUC'].iloc[0],
        'Test AUC-ROC': res[name]['test_metrics']['auc_roc'],
        'Best at experiment': int(kept.loc[kept['CV_AUC'].idxmax(), 'experiment']),
        'Kept': int((df['status'] == 'keep').sum()),
        'Discarded': int((df['status'] == 'discard').sum()),
        'Crashed': int((df['status'] == 'crash').sum()),
    })
pd.DataFrame(start).set_index('Approach').round(4)

In [ ]:
conv_cv = res['Conventional XGBoost']['cv']['best_mean_auc']

fig, ax = plt.subplots(figsize=(9, 5))
for name, df in logs.items():
    ax.step(df['experiment'], df['best_so_far'], where='post', color=COLORS[name], lw=2.4, label=name)
ax.axhline(conv_cv, color='#444', ls='--', lw=1, label='Conventional CV')
ax.set_xlabel('Experiment')
ax.set_ylabel('Best CV AUC so far')
ax.set_title('Best CV AUC so far, by experiment')
ax.grid(alpha=0.25)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()